# Event Ticket Sales Analysis

## Objective

Explore changes in customer demand from 1 June to 31 October 2024 and identify questions for the product team.

## Analysis tasks

1. Load the order, event, and exchange-rate datasets; inspect their size, structure, and missing values.
2. Prepare dates and categories, review revenue distributions and duplicate records, join the datasets, and create derived features.
3. Compare monthly order volume and summer/autumn order shares by event category, device type, and age restriction.
4. Explore daily autumn activity, orders per active buyer, weekday patterns, regions, and ticket partners.
5. Compare device segments and summarise the findings.

The filtering steps below document the rules used in the analysis; they should not be interpreted as proof that every excluded observation is an error.

## Data dictionary

### Orders — `final_tickets_orders_df.csv`

- `order_id`: unique order identifier.
- `user_id`: user identifier.
- `created_dt_msk`: order date in Moscow time.
- `created_ts_msk`: order timestamp in Moscow time.
- `event_id`: event identifier used to join the event dataset.
- `cinema_circuit`: cinema chain, where applicable.
- `age_limit`: event age restriction.
- `currency_code`: payment currency, such as RUB or KZT.
- `device_type_canonical`: ordering device, such as mobile or desktop.
- `revenue`: revenue attributed to the order.
- `service_name`: ticket partner.
- `tickets_count`: number of tickets in the order.
- `total`: total order value.

### Events — `final_tickets_events_df.csv`

- `event_id`: unique event identifier.
- `event_name`: event name or code.
- `event_type_description`: detailed event category.
- `event_type_main`: main event category.
- `organizers`: event organisers.
- `region_name`, `city_name`, `city_id`: event geography.
- `venue_id`, `venue_name`, `venue_address`: venue information.

### Exchange rates — `final_tickets_tenge_df.csv`

- `nominal`: currency units represented by the rate, 100 KZT in this dataset.
- `data`: rate date.
- `curs`: exchange rate in RUB.
- `cdx`: currency code.

# 1. Load the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_ind, mannwhitneyu, shapiro, levene
import statsmodels.api as sm
from statsmodels.stats.proportion import proportions_ztest

In [ ]:
orders = pd.read_csv('/datasets/final_tickets_orders_df.csv')
events = pd.read_csv('/datasets/final_tickets_events_df.csv')
tenge = pd.read_csv('/datasets/final_tickets_tenge_df.csv')

In [ ]:
print(f"Orders: {orders.shape[0]:,} rows, {orders.shape[1]} columns")
print(f"Events: {events.shape[0]:,} rows, {events.shape[1]} columns")
print(f"KZT exchange rates: {tenge.shape[0]:,} rows, {tenge.shape[1]} columns")

In [ ]:
print("=== ORDERS DATASET ===")
orders.info()

print("=== EVENTS DATASET ===")
events.info()

print("=== KZT EXCHANGE-RATE DATASET ===")
tenge.info()

The original inspection recorded 290,849 orders, 22,427 events, and 357 exchange-rate observations. There were 21,940 missing values in `days_since_prev`, which can occur for first purchases.

Date fields need conversion to datetime. Further checks cover event matches, exchange-rate coverage, revenue values, and duplicate orders.

# 2. Prepare the data

In [ ]:
orders['created_dt_msk'] = pd.to_datetime(orders['created_dt_msk'])
orders['created_ts_msk'] = pd.to_datetime(orders['created_ts_msk'])
tenge['data'] = pd.to_datetime(tenge['data'])

In [ ]:
print("Device types:", orders['device_type_canonical'].unique())
print("Currencies:", orders['currency_code'].unique())
print("Age restrictions:", sorted(orders['age_limit'].unique()))

In [ ]:
rub_revenue = orders[orders['currency_code'] == 'rub']['revenue']
print(rub_revenue.describe())

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(rub_revenue, bins=50, edgecolor='black')
plt.title('Revenue distribution in RUB (up to the 99th percentile)')
plt.xlabel('Revenue')
plt.ylabel('Frequency')
plt.show()

RUB revenue has a long right tail and includes negative values.

In [ ]:
kzt_revenue = orders[orders['currency_code'] == 'kzt']['revenue']
print(kzt_revenue.describe())

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(kzt_revenue, bins=50, edgecolor='black')
plt.title('Revenue distribution in KZT')
plt.xlabel('Revenue')
plt.ylabel('Frequency')
plt.show()

In [ ]:
rub_mask = orders['currency_code'] == 'rub'
rub_5 = orders[rub_mask]['revenue'].quantile(0.05)
rub_95 = orders[rub_mask]['revenue'].quantile(0.95)

orders_rub_clean = orders[
    (orders['currency_code'] == 'rub') &
    (orders['revenue'] >= rub_5) &
    (orders['revenue'] <= rub_95)
].copy()

In [ ]:
kzt_mask = orders['currency_code'] == 'kzt'
kzt_95 = orders[kzt_mask]['revenue'].quantile(0.95)

orders_kzt_clean = orders[
    (orders['currency_code'] == 'kzt') &
    (orders['revenue'] <= kzt_95)
].copy()

In [ ]:
orders_clean = pd.concat([orders_rub_clean, orders_kzt_clean], ignore_index=True)
print(f"\nOrders after filtering: {len(orders_clean):,}")
print(f"Total removed: {len(orders) - len(orders_clean):,}")

In [ ]:
rub_revenue = orders_clean[orders_clean['currency_code'] == 'rub']['revenue']
plt.figure(figsize=(10, 5))
plt.hist(rub_revenue, bins=50, edgecolor='black')
plt.title('Revenue distribution in RUB (up to the 99th percentile)')
plt.xlabel('Revenue')
plt.ylabel('Frequency')
plt.show()

Revenue records have been filtered using the rules in the preceding cells.

In [ ]:
print("Exact duplicates:", orders_clean.duplicated().sum())

In [ ]:
implicit_dupes = orders_clean.groupby(['user_id', 'event_id', 'created_dt_msk', 'tickets_count', 'total']).size()
implicit_dupes_count = (implicit_dupes > 1).sum()
print(f"Potential duplicates: {implicit_dupes_count}")

In [ ]:
print(f"Before removing potential duplicates: {len(orders_clean):,}")
orders_clean = orders_clean.drop_duplicates(subset=['user_id', 'event_id', 'created_dt_msk', 'tickets_count', 'total'], keep='first')
print(f"After removal: {len(orders_clean):,}")

Duplicate records have been removed using the selected key.

In [ ]:
full_df = orders_clean.merge(events, on='event_id', how='left')
print(f"Joined DataFrame: {full_df.shape[0]:,} rows, {full_df.shape[1]} columns")

In [ ]:
unmatched = full_df[full_df['event_name'].isna()]
print(f"Orders without an event match: {len(unmatched):,}")

In [ ]:
full_df = full_df[full_df['event_name'].notna()].copy()

In [ ]:
full_df = full_df.merge(tenge[['data', 'curs']], left_on='created_dt_msk', right_on='data', how='left')
rub_mask = full_df['currency_code'] == 'rub'
kzt_mask = full_df['currency_code'] == 'kzt'

full_df.loc[rub_mask, 'revenue_rub'] = full_df.loc[rub_mask, 'revenue']
full_df.loc[kzt_mask, 'revenue_rub'] = full_df.loc[kzt_mask, 'revenue'] / 100 * full_df.loc[kzt_mask, 'curs']

Revenue has been converted to RUB using the exchange-rate dataset.

In [ ]:
full_df['one_ticket_revenue_rub'] = full_df['revenue_rub'] / full_df['tickets_count']
full_df['month'] = full_df['created_dt_msk'].dt.month
def get_season(month):
    if month in [6, 7, 8]:
        return 'summer'
    elif month in [9, 10, 11]:
        return 'autumn'
    elif month in [1, 2, 12]:
        return 'winter'
    else:
        return 'spring'

full_df['season'] = full_df['month'].apply(get_season)

In [ ]:
full_df.info()

## Data preparation summary

Date fields were converted to datetime. RUB revenue was filtered between the 5th and 95th percentiles, while KZT revenue was capped at its 95th percentile. Duplicate checks used `user_id`, `event_id`, `created_dt_msk`, `tickets_count`, and `total`.

Orders were joined to events; the original analysis removed 24 records without an event match. KZT revenue was converted to RUB using the supplied exchange rates.

Derived fields:

- `revenue_rub`: order revenue in RUB.
- `one_ticket_revenue_rub`: platform revenue per ticket within an order.
- `month`: order month.
- `season`: calendar season.

The processed sample contains 214,408 of the 290,849 source orders, or approximately 73.7%. Results therefore depend on the cleaning and duplicate-handling rules.

# 3. Exploratory analysis

In [ ]:
monthly_orders = full_df.groupby('month').size().reset_index(name='orders_count')
print(monthly_orders)

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(monthly_orders['month'], monthly_orders['orders_count'], color='steelblue', edgecolor='black')
plt.title('Monthly orders')
plt.xlabel('Month')
plt.ylabel('Number of orders')
plt.xticks([6,7,8,9,10], ['June', 'July', 'August', 'September', 'October'])
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
summer_df = full_df[full_df['season'] == 'summer'].copy()
autumn_df = full_df[full_df['season'] == 'autumn'].copy()

print(f"Summer orders: {len(summer_df):,}")
print(f"Autumn orders: {len(autumn_df):,}")

In [ ]:
summer_event_pct = summer_df['event_type_main'].value_counts(normalize=True) * 100
autumn_event_pct = autumn_df['event_type_main'].value_counts(normalize=True) * 100

event_comparison = pd.DataFrame({
    'Summer (%)': summer_event_pct,
    'Autumn (%)': autumn_event_pct
}).fillna(0).sort_values('Autumn (%)', ascending=False)

print(event_comparison)

In [ ]:
plt.figure(figsize=(12, 6))
event_comparison.plot(kind='bar')
plt.title('Order share by event category: summer vs autumn', fontsize=14)
plt.xlabel('Event category')
plt.ylabel('Share (%)')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Season')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
summer_device_pct = summer_df['device_type_canonical'].value_counts(normalize=True) * 100
autumn_device_pct = autumn_df['device_type_canonical'].value_counts(normalize=True) * 100

device_comparison = pd.DataFrame({
    'Summer (%)': summer_device_pct,
    'Autumn (%)': autumn_device_pct
})

print(device_comparison)

In [ ]:
summer_age_pct = summer_df['age_limit'].value_counts(normalize=True).sort_index() * 100
autumn_age_pct = autumn_df['age_limit'].value_counts(normalize=True).sort_index() * 100

age_comparison = pd.DataFrame({
    'Summer (%)': summer_age_pct,
    'Autumn (%)': autumn_age_pct
}).fillna(0)

print(age_comparison)

In [ ]:
summer_avg_ticket = summer_df.groupby('event_type_main')['one_ticket_revenue_rub'].mean()
autumn_avg_ticket = autumn_df.groupby('event_type_main')['one_ticket_revenue_rub'].mean()

ticket_comparison = pd.DataFrame({
    'Summer': summer_avg_ticket,
    'Autumn': autumn_avg_ticket
}).fillna(0)

ticket_comparison['Change (%)'] = ((ticket_comparison['Autumn'] - ticket_comparison['Summer']) / ticket_comparison['Summer'] * 100).fillna(0)

print(ticket_comparison.sort_values('Change (%)', ascending=False).round(2))

In [ ]:
plt.figure(figsize=(12, 6))
ticket_sorted = ticket_comparison.sort_values('Autumn', ascending=False)
ticket_sorted[['Summer', 'Autumn']].plot(kind='bar')
plt.title('Mean platform revenue per ticket by event category')
plt.xlabel('Event category')
plt.ylabel('Revenue per ticket (RUB)')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Season')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In the processed sample, theatre and sports account for a larger autumn share of orders, while the concert share declines. Mobile devices remain the main source of orders.

`revenue_rub / tickets_count` measures platform revenue per ticket within an order. It is not the buyer's ticket price. Averaging this ratio across orders is different from dividing total revenue by total tickets.

In [ ]:
autumn_df = full_df[full_df['season'] == 'autumn'].copy()

In [ ]:
daily_autumn = autumn_df.groupby(autumn_df['created_dt_msk'].dt.date).agg({
    'order_id': 'count',
    'user_id': 'nunique',
    'one_ticket_revenue_rub': 'mean'
}).reset_index()

daily_autumn.columns = ['date', 'orders_count', 'dau', 'avg_ticket_price']
daily_autumn['orders_per_user'] = daily_autumn['orders_count'] / daily_autumn['dau']
print(daily_autumn.head(10))

In [ ]:
plt.figure(figsize=(15, 20))

plt.subplot(4, 1, 1)
plt.plot(daily_autumn['date'], daily_autumn['orders_count'], linewidth=1.5, color='steelblue')
plt.title('Daily orders: autumn 2024')
plt.ylabel('Number of orders')
plt.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45)

plt.subplot(4, 1, 2)
plt.plot(daily_autumn['date'], daily_autumn['dau'], linewidth=1.5, color='green')
plt.title('Daily active buyers')
plt.ylabel('Users')
plt.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45)

plt.subplot(4, 1, 3)
plt.plot(daily_autumn['date'], daily_autumn['orders_per_user'], linewidth=1.5, color='orange')
plt.title('Orders per active buyer')
plt.ylabel('Orders per user')
plt.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45)

plt.subplot(4, 1, 4)
plt.plot(daily_autumn['date'], daily_autumn['avg_ticket_price'], linewidth=1.5, color='red')
plt.title('Mean platform revenue per ticket')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Revenue per ticket (RUB)')
plt.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
autumn_df['weekday_name'] = autumn_df['created_dt_msk'].dt.day_name()

weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_rus = {
    'Monday': 'Mon', 'Tuesday': 'Tue', 'Wednesday': 'Wed', 'Thursday': 'Thu',
    'Friday': 'Fri', 'Saturday': 'Sat', 'Sunday': 'Sun'
}

daily_stats = autumn_df.groupby('weekday_name').agg({
    'order_id': 'count',
    'user_id': 'nunique',
    'one_ticket_revenue_rub': 'mean'
}).reindex(weekday_order)

daily_stats.index = daily_stats.index.map(weekday_rus)
daily_stats.columns = ['Orders', 'Users', 'Mean revenue per ticket']
daily_stats['Orders per user'] = daily_stats['Orders'] / daily_stats['Users']

print(daily_stats.round(2))

Autumn data shows an increase in orders and active buyers. Quantitative comparisons should use aligned dates and comparable observation periods.

`orders_count / dau` measures daily orders per active buyer, rather than tickets per visit. Weekday comparisons also depend on the number of observed days and the time trend.

Differences in revenue per ticket may reflect the mix of events. They do not establish customer purchasing power or profitability.

In [ ]:
region_stats = full_df.groupby('region_name').agg({
    'event_id': 'nunique',
    'order_id': 'count'
}).reset_index()
region_stats.columns = ['Region', 'Events', 'Orders']
region_stats = region_stats.sort_values('Orders', ascending=False)

print("TOP 10 REGIONS BY ORDERS:")
print(region_stats.head(10).to_string(index=False))

print("\nTOP 5 REGIONS BY EVENT VARIETY:")
print(region_stats.sort_values('Events', ascending=False).head(5).to_string(index=False))

In [ ]:
region_stats = full_df.groupby('region_name').agg({
    'event_id': 'nunique',
    'order_id': 'count'
}).reset_index()
region_stats.columns = ['Region', 'Events', 'Orders']
region_stats = region_stats.sort_values('Orders', ascending=False)

top10_regions = region_stats.head(10)

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(top10_regions['Region'], top10_regions['Orders'])
plt.title('Top 10 regions by orders')
plt.xlabel('Orders')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("Top 10 regions by orders:")
print(top10_regions[['Region', 'Orders']].to_string(index=False))

In [ ]:
top10_events = region_stats.sort_values('Events', ascending=False).head(10)

plt.figure(figsize=(10, 6))
plt.barh(top10_events['Region'], top10_events['Events'], color='green')
plt.title('Top 10 regions by unique events')
plt.xlabel('Number of events')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("Top 10 regions by event variety:")
print(top10_events[['Region', 'Events']].to_string(index=False))

In [ ]:
partner_stats = full_df.groupby('service_name').agg({
    'event_id': 'nunique',
    'order_id': 'count',
    'revenue_rub': 'sum'
}).reset_index()
partner_stats.columns = ['Partner', 'Events', 'Orders', 'Revenue']
partner_stats = partner_stats.sort_values('Orders', ascending=False)

print("\nTOP 10 TICKET PARTNERS:")
print(partner_stats.head(10).to_string(index=False))

In the original calculation, the largest region accounts for about 31% of orders, the top three regions for about 53%, and the five largest ticket partners for about 77% of orders in the processed sample.

These shares describe the supplied platform data, rather than the entire ticket market.

## Device comparison — orders per user

H₀: the mean number of orders per mobile user is no greater than the mean for desktop users.

H₁: the mean number of orders per mobile user is greater.

In [ ]:
user_orders = autumn_df.groupby(['user_id', 'device_type_canonical']).size().reset_index(name='orders')

mobile_ord = user_orders[user_orders['device_type_canonical'] == 'mobile']['orders']
desktop_ord = user_orders[user_orders['device_type_canonical'] == 'desktop']['orders']

print("Orders per user:")
print(f"Mobile — mean: {mobile_ord.mean():.2f}, median: {mobile_ord.median():.2f}")
print(f"Desktop — mean: {desktop_ord.mean():.2f}, median: {desktop_ord.median():.2f}")

In [ ]:
alpha = 0.05

stat1, p1 = mannwhitneyu(mobile_ord, desktop_ord, alternative='greater')
print(f"Significance level: {alpha}")
print(f"p-value = {p1:.10f}")

if p1 < alpha:
    print("Reject H0 in the specified comparison: mobile users place more orders")
else:
    print("Do not reject H0 at the specified significance level")

## Device comparison — time between orders

H₀: the mean interval between orders for mobile users is no greater than for desktop users.

H₁: the mean interval between orders for mobile users is greater.

In [ ]:
mobile_gap = autumn_df[autumn_df['device_type_canonical'] == 'mobile']['days_since_prev'].dropna()
desktop_gap = autumn_df[autumn_df['device_type_canonical'] == 'desktop']['days_since_prev'].dropna()

print(f"Time between orders (days):")
print(f"Mobile — mean: {mobile_gap.mean():.1f}, median: {mobile_gap.median():.1f}")
print(f"Desktop — mean: {desktop_gap.mean():.1f}, median: {desktop_gap.median():.1f}")

In [ ]:
stat_levene, p_levene = levene(mobile_gap, desktop_gap)
print(f"Levene test: p-value = {p_levene:.6f}")

In [ ]:
t_stat, p_t = ttest_ind(mobile_gap, desktop_gap, alternative='greater', equal_var=False)
print(f"\nt-test: p-value = {p_t:.10f}")
if p_t < alpha:
    print("Reject H0 in the specified comparison: mobile users have longer order intervals")
else:
    print("Do not reject H0 at the specified significance level")

In [ ]:
stat2, p2 = mannwhitneyu(mobile_gap, desktop_gap, alternative='greater')
print(f"Significance level: {alpha}")
print(f"p-value = {p2:.10f}")

if p2 < alpha:
    print("Reject H0 in the specified comparison: mobile users have longer order intervals")
else:
    print("Do not reject H0 at the specified significance level")

The original analysis found differences between device segments. Interpretation requires checking whether users occur in both groups and whether repeated intervals from one user are dependent. These comparisons do not establish a causal device effect. The Mann–Whitney test is not, by itself, a test of equality of means.

# Conclusions

In the processed sample, the autumn share of theatre and sports orders increases, while the concert share decreases. About 53% of orders come from the three largest regions. Mobile devices account for most orders.

The analysis provides a view of demand composition and highlights segments for further investigation. Device comparisons require attention to the user-level unit of analysis. Orders per buyer, tickets per order, ticket price, and platform revenue per ticket are different measures.

The processed sample contains 214,408 orders compared with 290,849 in the original extract. Findings depend on the cleaning rules. The SQL dashboard and Python sample require aligned filters before their totals can be compared.

Potential next questions include the mobile purchase journey and the event mix behind seasonal changes. Pricing or promotion decisions require additional evidence on costs and margins.